## Setup

In [ ]:
GITHUB_URL = "https://github.com/arjuns07/cs166-final-project"
REPO_DIR   = "/content/cs166-final-project"

import os, subprocess

if os.path.isdir(REPO_DIR):
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    from google.colab import userdata
    token = "TODO"
    repo_slug = GITHUB_URL.replace("https://github.com/", "")
    auth_url = f"https://{token}@github.com/{repo_slug}"
    result = subprocess.run(["git", "clone", auth_url, REPO_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)

Cloning into '/content/cs166-final-project'...



In [ ]:
import shutil, subprocess, sys

if shutil.which("uv") is None:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)

pyver = f"{sys.version_info.major}.{sys.version_info.minor}"  # Colab kernel's version
r = subprocess.run(["uv", "sync", "--python", pyver], cwd=REPO_DIR,
                   capture_output=True, text=True)
print(r.stdout or r.stderr)
r.check_returncode()

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 56 packages in 12ms
Prepared 54 packages in 12.07s
Installed 54 packages in 150ms
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + gsplat==1.5.3
 + jaxtyping==0.3.10
 + jinja2==3.1.6
 + kiwisolver==1.5.0
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + mdurl==0.1.2
 + mpmath==1.3.0
 + networkx==3.6.1
 + ninja==1.13.0
 + numpy==2.4.6
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.20.0.48
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 + nvidia-cusolver==12.0.4.66
 + nvidia-cusparse==12.6.3.3
 + nvidia-cusparselt-cu13==0.8.1
 + nvidia-nccl-cu13==2.29.7
 + nvidia-nvjitlink==13.0.88
 + nvidia-nvshmem-cu13==3.4.5


In [ ]:
import sys, os, glob
from pathlib import Path

REPO_ROOT = Path(REPO_DIR)
os.chdir(REPO_ROOT)

venv_site_packages = glob.glob(str(REPO_ROOT / ".venv/lib/python*/site-packages"))
assert venv_site_packages, "venv not found — did uv sync succeed?"
for sp in venv_site_packages:
    if sp not in sys.path:
        sys.path.insert(0, sp)

for p in [str(REPO_ROOT), str(REPO_ROOT / "basics"), str(REPO_ROOT / "3dgs-python")]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("cwd =", os.getcwd())
print("sys.path[:4] =", sys.path[:4])


cwd = /content/cs166-final-project
sys.path[:4] = ['/content/cs166-final-project/3dgs-python', '/content/cs166-final-project/basics', '/content/cs166-final-project', '/content/cs166-final-project/.venv/lib/python3.12/site-packages']


In [ ]:
!pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 127.6 MB/s eta 0:00:00


In [ ]:
!cd {REPO_DIR} && uv add torchvision
!cd {REPO_DIR} && uv sync

Resolved 57 packages in 302ms
Prepared 1 package in 88ms
Installed 1 package in 4ms
 + torchvision==0.27.0
Resolved 57 packages in 0.59ms
Checked 55 packages in 0.40ms


## Download

In [ ]:
from google.colab import drive
import os
import subprocess

# Mount Drive
drive.mount('/content/drive')

# Config
DRIVE_DIR = '/content/drive/MyDrive/datasets/mipnerf360'
ZIP_PATH  = os.path.join(DRIVE_DIR, 'garden.zip')
URL       = 'https://huggingface.co/nerfbaselines/nerfbaselines/resolve/main/gaussian-splatting/mipnerf360/garden.zip'

os.makedirs(DRIVE_DIR, exist_ok=True)

# Download only if not cached
if not os.path.exists(ZIP_PATH):
    print("Downloading...")
    subprocess.run(['wget', '-O', ZIP_PATH, URL], check=True)
else:
    print(f"Found cached file: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e9:.2f} GB)")

Mounted at /content/drive
Found cached file: /content/drive/MyDrive/datasets/mipnerf360/garden.zip (5.85 GB)


## Render scene

In [ ]:
import numpy as np
import torch
from pathlib import Path

In [ ]:
CAM_DIR = Path("/content/drive/MyDrive/datasets/mipnerf360/predictions/cameras/")
PLY_PATH = "/content/drive/MyDrive/datasets/mipnerf360/checkpoint/point_cloud/iteration_30000/point_cloud.ply"
DEVICE = "cuda"

In [ ]:
def load_camera(npz_path):
    cam = np.load(npz_path)
    R_t = cam['poses']  # (3, 4)
    P = np.eye(4)
    P[:3, :] = R_t
    fx, fy, cx, cy = cam['intrinsics']
    w, h = cam['image_sizes']
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    return torch.tensor(P, dtype=torch.float32), torch.tensor(K, dtype=torch.float32), int(w), int(h)

cam_paths = sorted(CAM_DIR.glob("*.npz"))
camera_view_names = [str(p).split("/")[-1].split(".")[0] for p in list(cam_paths)]

extrinsics, Ks, ws, hs = zip(*[load_camera(p) for p in cam_paths])

K = Ks[0].to(DEVICE)
w, h = ws[0], hs[0]
all_P = torch.stack(list(extrinsics)).to(DEVICE)

In [ ]:
from gaussian_scene import GaussianScene
scene = GaussianScene(max_sh_degree=3, device=DEVICE)
scene.init_full_scene_from_ply(PLY_PATH)

In [ ]:
scene.calc_n_gaussians()

5854520

In [ ]:
!rm -rf renders

In [ ]:
from render import render_all_camera_views
os.mkdir("renders")
os.mkdir("deltas")
save_to_file_paths = [f"renders/{view_name}.png" for view_name in camera_view_names]

render_all_camera_views(
    scene=scene,
    all_camera_extrinsics=all_P,
    camera_intrinsics_K=K,
    img_dims=(h, w),
    device=DEVICE,
    save_to_files=save_to_file_paths,
)

In [ ]:
from google.colab import files
!zip -r renders.zip renders
files.download("renders.zip")

  adding: renders/ (stored 0%)
  adding: renders/DSC08116.png (deflated 0%)
  adding: renders/DSC08052.png (deflated 0%)
  adding: renders/DSC07980.png (deflated 0%)
  adding: renders/DSC07988.png (deflated 0%)
  adding: renders/DSC08004.png (deflated 0%)
  adding: renders/DSC08084.png (deflated 0%)
  adding: renders/DSC08132.png (deflated 0%)
  adding: renders/DSC08036.png (deflated 0%)
  adding: renders/DSC07996.png (deflated 0%)
  adding: renders/DSC08108.png (deflated 0%)
  adding: renders/DSC07964.png (deflated 0%)
  adding: renders/DSC08092.png (deflated 0%)
  adding: renders/DSC08060.png (deflated 0%)
  adding: renders/DSC07972.png (deflated 0%)
  adding: renders/DSC08012.png (deflated 0%)
  adding: renders/DSC08020.png (deflated 0%)
  adding: renders/DSC08068.png (deflated 0%)
  adding: renders/DSC08028.png (deflated 0%)
  adding: renders/DSC08100.png (deflated 0%)
  adding: renders/DSC07956.png (deflated 0%)
  adding: renders/DSC08124.png (deflated 0%)
  adding: renders/DSC080

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from torchvision.utils import save_image
from torchvision.io import read_image


REF_VIEW_DIR = Path("/content/drive/MyDrive/datasets/mipnerf360/predictions/color/")
GT_VIEW_DIR = Path("/content/drive/MyDrive/datasets/mipnerf360/images_4/")

for view_name in camera_view_names:
    gt = read_image(f"{GT_VIEW_DIR}/{view_name}.JPG").float()
    ref = read_image(f"{REF_VIEW_DIR}/{view_name}.png").float()
    ours = read_image(f"renders/{view_name}.png").float()

    psnr_ref_gt  = 10 * torch.log10(255**2 / ((gt - ref)**2).mean())
    psnr_ours_gt = 10 * torch.log10(255**2 / ((gt - ours)**2).mean())
    psnr_ours_ref = 10 * torch.log10(255**2 / ((ref - ours)**2).mean())
    print(f"{view_name}: ref:gt={psnr_ref_gt:.2f}  ours:gt={psnr_ours_gt:.2f}  ours:ref={psnr_ours_ref:.2f}")

    delta_ours_theirs = (ours - ref)
    save_image(delta_ours_theirs, f"deltas/{view_name}.png")


DSC07956: ref:gt=23.66  ours:gt=23.42  ours:ref=38.42
DSC07964: ref:gt=22.56  ours:gt=22.45  ours:ref=40.16
DSC07972: ref:gt=27.97  ours:gt=27.72  ours:ref=40.02
DSC07980: ref:gt=24.70  ours:gt=24.55  ours:ref=40.59
DSC07988: ref:gt=19.80  ours:gt=19.77  ours:ref=40.12
DSC07996: ref:gt=26.34  ours:gt=26.11  ours:ref=40.73
DSC08004: ref:gt=29.35  ours:gt=28.87  ours:ref=38.62
DSC08012: ref:gt=29.15  ours:gt=28.79  ours:ref=39.98
DSC08020: ref:gt=27.14  ours:gt=26.98  ours:ref=41.06
DSC08028: ref:gt=27.25  ours:gt=27.11  ours:ref=40.74
DSC08036: ref:gt=27.36  ours:gt=27.05  ours:ref=39.69
DSC08044: ref:gt=28.88  ours:gt=28.41  ours:ref=40.34
DSC08052: ref:gt=27.35  ours:gt=27.05  ours:ref=40.11
DSC08060: ref:gt=29.67  ours:gt=29.41  ours:ref=42.78
DSC08068: ref:gt=30.22  ours:gt=29.81  ours:ref=39.45
DSC08076: ref:gt=25.76  ours:gt=25.61  ours:ref=41.54
DSC08084: ref:gt=27.64  ours:gt=27.33  ours:ref=41.87
DSC08092: ref:gt=30.81  ours:gt=30.36  ours:ref=41.70
DSC08100: ref:gt=28.33  ours

In [ ]:
from google.colab import files
!zip -r deltas.zip deltas
files.download("deltas.zip")

  adding: deltas/ (stored 0%)
  adding: deltas/DSC08116.png (deflated 0%)
  adding: deltas/DSC08052.png (deflated 0%)
  adding: deltas/DSC07980.png (deflated 0%)
  adding: deltas/DSC07988.png (deflated 0%)
  adding: deltas/DSC08004.png (deflated 0%)
  adding: deltas/DSC08084.png (deflated 0%)
  adding: deltas/DSC08132.png (deflated 0%)
  adding: deltas/DSC08036.png (deflated 0%)
  adding: deltas/DSC07996.png (deflated 0%)
  adding: deltas/DSC08108.png (deflated 0%)
  adding: deltas/DSC07964.png (deflated 0%)
  adding: deltas/DSC08092.png (deflated 0%)
  adding: deltas/DSC08060.png (deflated 0%)
  adding: deltas/DSC07972.png (deflated 0%)
  adding: deltas/DSC08012.png (deflated 0%)
  adding: deltas/DSC08020.png (deflated 0%)
  adding: deltas/DSC08068.png (deflated 0%)
  adding: deltas/DSC08028.png (deflated 0%)
  adding: deltas/DSC08100.png (deflated 0%)
  adding: deltas/DSC07956.png (deflated 0%)
  adding: deltas/DSC08124.png (deflated 0%)
  adding: deltas/DSC08076.png (deflated 0%)
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>